# Graph-Based AML Detection with LineMVGNN

This notebook implements the PRD end-to-end: transaction-level graph
construction (NetworKit) -> feature engineering -> PyTorch Geometric export
-> LineMVGNN training -> evaluation -> model persistence -> inference ->
account-level risk aggregation -> dashboard export.

**Every heavy-compute section is wrapped in a `Timer(...)` context manager**
(see `timing_utils.py`). The last section of this notebook prints/plots a
full breakdown of where the run actually spent its time.

### A few things worth knowing before you run this

1. **No real data is bundled here.** `HI-Small_Trans.csv` / `LI-Small_Trans.csv`
   (the IBM AML dataset) aren't present in this environment, so Section 1
   generates a small synthetic dataset with the *same column schema* so the
   whole pipeline can be built, run, and timed. **To use real data**, just
   change the two file paths in Section 1 — nothing else changes.
2. **`LineMVGNN` isn't a published architecture with a fixed spec in the
   PRD** — only its purpose ("learn transaction relationships and
   money-flow structures") and the embedding strategy (Section 6) are
   specified. `model.py` implements it as a **Multi-View GNN**: a
   structural/topology view (GAT over the money-flow graph), an
   account-context view (GCN), and a graph-free transaction-attribute view
   (MLP), fused into the Section 10 classifier head. If you have a specific
   paper/architecture in mind, swap out `model.py` — everything else is
   architecture-agnostic.
3. **`torch_geometric.loader.NeighborLoader` needs `pyg-lib`/`torch-sparse`**,
   which ship as wheels from a separate package index
   (`data.pyg.org`) that this sandbox's network allowlist doesn't reach, and
   building them from source isn't practical here. `neighbor_sampling.py`
   re-implements the same idea (mini-batch sampling so you never load the
   full graph into memory/GPU) in plain Python/PyTorch using the
   predecessor adjacency lists from graph construction. **If your own
   environment can install `pyg-lib`/`torch-sparse`**, swap in the real
   `NeighborLoader` against the exact same `Data` object — no other code
   changes needed (see the comment at the top of `neighbor_sampling.py`).
4. This notebook expects the accompanying `.py` modules
   (`timing_utils.py`, `data_processing.py`, `graph_construction.py`,
   `feature_engineering.py`, `pyg_export.py`, `neighbor_sampling.py`,
   `model.py`, `train.py`, `aggregation.py`) to sit in the **same folder**
   as this notebook.


In [4]:
# If a package is missing in your environment, uncomment:
# !pip install torch torch_geometric networkit pandas numpy scikit-learn plotly streamlit nbformat

import time
import numpy as np
import pandas as pd
import torch
import plotly.express as px


from src.synthetic_data import generate_synthetic_transactions
from src.data_processing import load_and_clean, vocab_sizes
from src.graph_construction import build_transaction_graph
from src.feature_engineering import engineer_all_features
from src.pyg_export import build_pyg_data, NUMERIC_FEATURE_COLUMNS
from src.model import LineMVGNN
from src.train import train_model, run_inference, compute_classification_metrics, print_metrics_report
from src.aggregation import build_transaction_view, aggregate_accounts


torch.manual_seed(0)
np.random.seed(0)


ModuleNotFoundError: No module named 'torch'

## 1. Dataset

PRD Section 3: training data is `HI-Small_Trans.csv`, testing data is
`LI-Small_Trans.csv`, target label `Is Laundering` (binary).

Swap `TRAIN_CSV` / `TEST_CSV` below for the real IBM-AML files when you
have them — the rest of the notebook is unchanged either way.

In [ ]:
TRAIN_CSV = "data/HI-Small_Trans.csv"  
TEST_CSV = "data/LI-Small_Trans.csv"   

import os
if not (os.path.exists(TRAIN_CSV) and os.path.exists(TEST_CSV)):
    print("Real IBM-AML files not found -- generating a synthetic stand-in dataset "
          "with the same schema so the pipeline can run end-to-end.")
    generate_synthetic_transactions(n_accounts=1500, n_transactions=40000, seed=42)\
        .to_csv(TRAIN_CSV, index=False)
    generate_synthetic_transactions(n_accounts=1200, n_transactions=15000, seed=99)\
        .to_csv(TEST_CSV, index=False)

pd.read_csv(TRAIN_CSV).head()


Real IBM-AML files not found -- generating a synthetic stand-in dataset with the same schema so the pipeline can run end-to-end.


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022-09-01 00:03:37.660963977,25,ACC100794,36,ACC100955,2268.04,Yen,2268.04,Yen,Cheque,0
1,2022-09-01 00:06:22.640417406,44,ACC100390,25,ACC100953,99.17,Yen,99.17,Yen,Reinvestment,0
2,2022-09-01 00:10:29.276885350,30,ACC100343,4,ACC100126,2660.08,Euro,2660.08,Euro,Credit Card,0
3,2022-09-01 00:12:54.889506790,47,ACC100483,0,ACC101278,87.59,Saudi Riyal,87.59,Bitcoin,Cheque,0
4,2022-09-01 00:16:32.754439972,30,ACC100896,6,ACC100257,1175.10,Yen,1175.10,Yen,Cash,0


## 2. Data Cleaning & Normalization  *(timed: `01.`)*

Categorical vocabularies (account / bank / payment format / currency) are
fit on the **training** file only, then reused on the test file (with an
explicit "unknown" bucket for anything only seen at test time) — otherwise
train/test embedding indices wouldn't refer to the same accounts.

In [3]:
df_train, vocabs = load_and_clean(TRAIN_CSV)
df_test, _ = load_and_clean(TEST_CSV, vocabs=vocabs)

print(f"Train: {len(df_train):,} transactions   |   Test: {len(df_test):,} transactions")
print(f"Laundering rate -- train: {df_train['label'].mean():.3%}   test: {df_test['label'].mean():.3%}")
vocab_sizes(vocabs)


[TIMER] 01. Data Loading & Normalization                 0.130s
[TIMER] 01. Data Loading & Normalization                 0.054s
Train: 40,000 transactions   |   Test: 15,000 transactions
Laundering rate -- train: 4.500%   test: 4.500%


{'n_accounts': 1501, 'n_banks': 61, 'n_payment_formats': 7, 'n_currencies': 7}

## 3. Transaction Graph Construction (NetworKit)  *(timed: `02.`)*

Each transaction is a node. PRD Section 5's edge rule: `T1 -> T2` iff
`receiver(T1) == sender(T2)`, `timestamp(T2) > timestamp(T1)`, within a
10-day window, capped at the 15 chronologically-nearest successors per
node (Constraint 3, prevents hub accounts from blowing up graph density).

In [ ]:
(
    g_train,
    src_train,
    dst_train,
    pred_indptr_train,
    pred_indices_train,
    succ_indptr_train,
    succ_indices_train,
) = build_transaction_graph(df_train)

(
    g_test,
    src_test,
    dst_test,
    pred_indptr_test,
    pred_indices_test,
    succ_indptr_test,
    succ_indices_test,
) = build_transaction_graph(df_test)

print(f"Train graph: {g_train.numberOfNodes():,} nodes, {g_train.numberOfEdges():,} edges "
      f"(avg out-degree {g_train.numberOfEdges()/g_train.numberOfNodes():.2f})")
print(f"Test graph:  {g_test.numberOfNodes():,} nodes, {g_test.numberOfEdges():,} edges")


[TIMER] 02. Transaction Graph Construction (NetworKit)    0.435s


[TIMER] 02. Transaction Graph Construction (NetworKit)    0.288s
Train graph: 40,000 nodes, 164,225 edges (avg out-degree 4.11)
Test graph:  15,000 nodes, 29,062 edges


## 4. Feature Engineering  *(timed: `03a.`-`03d.`)*

Broken into 4 independently-timed sub-stages (Sections 7-9 of the PRD):
temporal, pair-history, transaction-flow (graph-topology), and
account-context (sender/receiver behavioral profiles — causal/non-leaky,
computed in a single chronological streaming pass).

In [ ]:
df_train = engineer_all_features(
    df_train,
    pred_indptr_train,
    pred_indices_train,
    succ_indptr_train,
    succ_indices_train,
    src_train,
    dst_train
)
df_test = engineer_all_features(
    df_train,
    pred_indptr_train,
    pred_indices_train,
    succ_indptr_train,
    succ_indices_train,
    src_train,
    dst_train
)

print(f"Engineered feature count: {len(NUMERIC_FEATURE_COLUMNS)} numeric + 7 categorical embedding inputs")
df_train.filter(regex="^(sender_|receiver_|pair_|fan_|relay_|short_cycle)").head()


[TIMER] 03a. Feature Engineering - Temporal Features     0.057s
[TIMER] 03b. Feature Engineering - Pair History Features    0.074s


[TIMER] 03c. Feature Engineering - Transaction Flow Features    0.524s


[TIMER] 03d. Feature Engineering - Account Context Features    1.501s
[TIMER] 03a. Feature Engineering - Temporal Features     0.010s
[TIMER] 03b. Feature Engineering - Pair History Features    0.020s
[TIMER] 03c. Feature Engineering - Transaction Flow Features    0.146s


[TIMER] 03d. Feature Engineering - Account Context Features    0.337s
Engineered feature count: 39 numeric + 7 categorical embedding inputs


,sender_idx,receiver_idx,pair_prior_txn_count,pair_total_amount_prior,pair_mean_amount_prior,pair_std_amount_prior,pair_max_amount_prior,pair_repeated_exact_amount_count,fan_in,fan_out,...,sender_velocity_1d,sender_velocity_10d,receiver_hist_outflow_total,receiver_hist_inflow_total,receiver_unique_counterparties,receiver_entropy,receiver_concentration,receiver_degree_balance,receiver_velocity_1d,receiver_velocity_10d
0,794,955,0,0.0,0.0,0.0,0.0,0,0,4,...,0,0,0.0,0.0,0,0.0,0.0,0.0,0,0
1,390,953,0,0.0,0.0,0.0,0.0,0,0,4,...,0,0,0.0,0.0,0,0.0,0.0,0.0,0,0
2,343,126,0,0.0,0.0,0.0,0.0,0,0,1,...,0,0,0.0,0.0,0,0.0,0.0,0.0,0,0
3,483,1278,0,0.0,0.0,0.0,0.0,0,0,7,...,0,0,0.0,0.0,0,0.0,0.0,0.0,0,0
4,896,257,0,0.0,0.0,0.0,0.0,0,0,2,...,0,0,0.0,0.0,0,0.0,0.0,0.0,0,0


## 5. Graph Export to PyTorch Geometric  *(timed: `04.`)*

One `StandardScaler` is fit on train and reused on test, so feature
scaling is consistent across both splits.

In [6]:
data_train, scaler = build_pyg_data(df_train, src_train, dst_train, fit_scaler=True)
data_test, _ = build_pyg_data(df_test, src_test, dst_test, scaler=scaler, fit_scaler=False)

print(data_train)
print(data_test)


[TIMER] 04. Graph Export to PyTorch Geometric            0.049s
[TIMER] 04. Graph Export to PyTorch Geometric            0.004s
Data(x=[40000, 39], edge_index=[2, 164225], y=[40000], num_nodes=40000, sender_idx=[40000], receiver_idx=[40000], from_bank_idx=[40000], to_bank_idx=[40000], payment_format_idx=[40000], payment_currency_idx=[40000], receiving_currency_idx=[40000])
Data(x=[15000, 39], edge_index=[2, 29062], y=[15000], num_nodes=15000, sender_idx=[15000], receiver_idx=[15000], from_bank_idx=[15000], to_bank_idx=[15000], payment_format_idx=[15000], payment_currency_idx=[15000], receiving_currency_idx=[15000])


## 6. Model: LineMVGNN  (PRD Sections 6 & 10)

See the design note at the top of this notebook / the docstring in
`model.py` for what each of the three "views" represents.

In [7]:
vs = vocab_sizes(vocabs)
model = LineMVGNN(
    numeric_dim=len(NUMERIC_FEATURE_COLUMNS),
    n_accounts=vs["n_accounts"],
    n_banks=vs["n_banks"],
    n_payment_formats=vs["n_payment_formats"],
    n_currencies=vs["n_currencies"],
    emb_dim=16, hidden_dim=64, num_layers=2, dropout=0.3,
)
print(model)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}")


LineMVGNN(
  (account_emb): Embedding(1501, 16)
  (bank_emb): Embedding(61, 8)
  (payfmt_emb): Embedding(7, 8)
  (currency_emb): Embedding(7, 8)
  (input_proj): Linear(in_features=111, out_features=64, bias=True)
  (gat_layers): ModuleList(
    (0-1): 2 x GATConv(64, 16, heads=4)
  )
  (gcn_layers): ModuleList(
    (0-1): 2 x GCNConv(64, 64)
  )
  (attr_mlp): Sequential(
    (0): Linear(in_features=64, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=64, bias=True)
  )
  (fusion): Linear(in_features=192, out_features=64, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (classifier): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=32, out_features=1, bias=True)
  )
)
Trainable parameters: 71,465


## 7. Training  (PRD Section 11)  *(timed: `05.`, with sub-timers `05a/05b/05c`)*

- **Sampling:** `SimpleNeighborLoader` (see the note at the top of this
  notebook re: `NeighborLoader`/`pyg-lib`).
- **Loss:** weighted `BCEWithLogitsLoss` (class-imbalance correction via
  `pos_weight`), equivalent to the spec'd weighted BCE + sigmoid.
- **Optimizer:** AdamW. **Scheduler:** `ReduceLROnPlateau` on validation PR-AUC.
- Validation split is the **last 15% chronologically** (not a random
  split) — appropriate for a temporal money-flow graph.

`05a`/`05b`/`05c` are cumulative *sub-totals inside* `05` (sampling vs.
forward/backward vs. validation), not separate top-level stages — see the
final timing report for how they relate to the rest of the pipeline.

In [8]:
model, history = train_model(
    model, data_train, preds_train,
    num_epochs=8, batch_size=512, num_neighbors=(15, 10),
    lr=1e-3, weight_decay=1e-4, val_frac=0.15,
)

fig = px.line(
    pd.DataFrame(history).reset_index().rename(columns={"index": "epoch"}),
    x="epoch", y=["train_loss", "val_loss"],
    title="Training / validation loss curves",
    labels={"value": "BCE loss", "variable": "split"},
)
fig.show()

fig2 = px.line(
    pd.DataFrame(history).reset_index().rename(columns={"index": "epoch"}),
    x="epoch", y="val_pr_auc", title="Validation PR-AUC by epoch",
)
fig2.show()


Train pos_weight (class-imbalance correction): 20.96 (1548 positive / 32452 negative)


Epoch  1/8  train_loss=0.9483  val_loss=0.4253  val_PR-AUC=0.7426  lr=1.00e-03


Epoch  2/8  train_loss=0.2520  val_loss=0.3354  val_PR-AUC=0.7675  lr=1.00e-03


Epoch  3/8  train_loss=0.2103  val_loss=0.3756  val_PR-AUC=0.7735  lr=1.00e-03


Epoch  4/8  train_loss=0.1818  val_loss=0.3642  val_PR-AUC=0.7812  lr=1.00e-03


Epoch  5/8  train_loss=0.1659  val_loss=0.5137  val_PR-AUC=0.7903  lr=1.00e-03


Epoch  6/8  train_loss=0.1559  val_loss=0.6188  val_PR-AUC=0.7837  lr=1.00e-03


Epoch  7/8  train_loss=0.1547  val_loss=0.5262  val_PR-AUC=0.7902  lr=1.00e-03


Epoch  8/8  train_loss=0.1365  val_loss=0.5983  val_PR-AUC=0.7870  lr=5.00e-04
[TIMER] 05. Model Training (total)                      65.147s


## 8. Evaluation Metrics on the Held-Out Test Set  (PRD Section 12)  *(timed: `06.`)*

Accuracy alone is insufficient given the class imbalance — PR-AUC and
Recall are the metrics that matter most for AML.

In [9]:
test_probs = run_inference(model, data_test, preds_test, batch_size=1024, num_neighbors=(15, 10))
test_y = data_test.y.numpy()

metrics = compute_classification_metrics(test_y, test_probs, threshold=0.5)
print_metrics_report(metrics)

cm = metrics["confusion_matrix"]
fig = px.imshow(
    cm, text_auto=True, color_continuous_scale="Blues",
    labels=dict(x="Predicted", y="Actual", color="Count"),
    x=["Legitimate (0)", "Laundering (1)"], y=["Legitimate (0)", "Laundering (1)"],
    title="Confusion matrix - test set",
)
fig.show()


[TIMER] 06. Model Inference                              0.210s

                      TEST SET METRICS                      
accuracy    : 0.9783
precision   : 0.7206
recall      : 0.8444
f1          : 0.7776
pr_auc      : 0.8635
Confusion matrix [rows=true, cols=pred] (0=legit, 1=laundering):
[[14104   221]
 [  105   570]]



## 9. Model Persistence  (PRD Section 13)

Weights are saved with `torch.save()` to `line_mvgnn_weights.pth` as
specified. The scaler + vocabularies are also persisted (via `pickle`)
since the inference pipeline needs the *exact* training-time feature
scaling and categorical-index mapping to produce consistent scores — this
is a practical addition beyond what Section 13 lists, but without it a
separately-loaded model can't be used safely on new data.

In [10]:
import pickle

torch.save(model.state_dict(), "line_mvgnn_weights.pth")
with open("preprocessing_state.pkl", "wb") as f:
    pickle.dump({"scaler": scaler, "vocabs": vocabs}, f)

print("Saved line_mvgnn_weights.pth and preprocessing_state.pkl")

# Reload sanity check
reloaded = LineMVGNN(
    numeric_dim=len(NUMERIC_FEATURE_COLUMNS),
    n_accounts=vs["n_accounts"], n_banks=vs["n_banks"],
    n_payment_formats=vs["n_payment_formats"], n_currencies=vs["n_currencies"],
)
reloaded.load_state_dict(torch.load("line_mvgnn_weights.pth"))
reloaded.eval()
probs_reloaded = run_inference(reloaded, data_test, preds_test, batch_size=1024, num_neighbors=(15, 10))
print("Max abs difference vs. original model's probabilities:", np.abs(test_probs - probs_reloaded).max())


Saved line_mvgnn_weights.pth and preprocessing_state.pkl


[TIMER] 06. Model Inference                              0.204s
Max abs difference vs. original model's probabilities: 0.0


## 10. Risk Ranking & Account Aggregation  (PRD Sections 14-16)  *(timed: `07.`)*

Transaction-level risk scores -> Transaction View -> Account View
(account risk = max risk score among any transaction it touched as
sender or receiver; alert fires on >=1 flagged transaction). These two
tables, plus the timing log, are exported for `dashboard.py`.

In [11]:
RISK_THRESHOLD = 0.5

transaction_view = build_transaction_view(df_test, test_probs, threshold=RISK_THRESHOLD)
account_view = aggregate_accounts(df_test, test_probs, threshold=RISK_THRESHOLD)

transaction_view.to_csv("transaction_view.csv", index=False)
account_view.to_csv("account_view.csv", index=False)

print(f"{transaction_view['Flagged'].sum():,} / {len(transaction_view):,} transactions flagged "
      f"at threshold {RISK_THRESHOLD}")
print(f"{account_view['Account Alert'].sum():,} / {len(account_view):,} accounts alerted")
transaction_view.head(10)


[TIMER] 07. Account Risk Aggregation                     0.015s
791 / 15,000 transactions flagged at threshold 0.5
736 / 1,200 accounts alerted


,Transaction ID,Risk Score,Sender,Receiver,Amount,Timestamp,Flagged
0,14148,0.999998,ACC100589,ACC100279,7334.36,2022-10-27 13:19:32.703698141,True
1,14855,0.999990,ACC100639,ACC100163,3221.17,2022-10-30 11:18:57.053997883,True
2,9423,0.999973,ACC100606,ACC100871,5224.44,2022-10-08 17:52:58.405884471,True
3,5793,0.999949,ACC100011,ACC100810,7973.20,2022-09-24 07:11:38.073359486,True
4,2259,0.999945,ACC100406,ACC100490,7726.15,2022-09-09 20:05:47.938134983,True
5,4113,0.999895,ACC100437,ACC100309,7604.44,2022-09-17 14:01:17.545206158,True
6,11045,0.999894,ACC100616,ACC101116,4789.45,2022-10-15 07:40:16.080370195,True
7,9264,0.999893,ACC100105,ACC100720,2694.74,2022-10-08 00:32:33.441440211,True
8,14498,0.999879,ACC101109,ACC100659,5138.11,2022-10-29 00:31:11.870628845,True
9,12585,0.999872,ACC100878,ACC100862,6359.12,2022-10-21 10:49:35.799399661,True


In [12]:
account_view.head(10)

,Account ID,Associated Risk Score,Mean Transaction Risk,Number of Flagged Transactions,Total Transactions,Counterparty Summary,Account Alert
0,ACC100279,0.999998,0.147669,3,26,25,True
1,ACC100589,0.999998,0.119782,4,33,31,True
2,ACC100163,0.999990,0.112595,3,29,28,True
3,ACC100639,0.999990,0.130574,4,30,28,True
4,ACC100606,0.999973,0.101112,3,29,28,True
5,ACC100871,0.999973,0.137558,4,28,26,True
6,ACC100810,0.999949,0.146911,4,27,24,True
7,ACC100011,0.999949,0.123691,4,32,30,True
8,ACC100406,0.999945,0.086429,2,23,20,True
9,ACC100490,0.999945,0.095321,2,23,21,True


## 11. Timing Report — Where Did the Time Go?

This is the answer to "what's taking up the most time": every heavy
section above (graph construction, each feature-engineering sub-stage,
PyG export, training — split into sampling vs. forward/backward vs.
validation — inference, and account aggregation) was wrapped in a
`Timer(...)`. The table/chart below are sorted slowest-first.

Note: `05a`/`05b`/`05c` are sub-totals **inside** `05. Model Training
(total)`, not additional top-level time — don't double-count them
against the other rows when eyeballing percentages.

In [13]:
timing_df = print_timing_report()
fig = plot_timing_breakdown(save_path="timing_breakdown.html")
fig.show()
save_timing_log("timing_log.json")
timing_df



                          PIPELINE TIMING BREAKDOWN                           
05. Model Training (total)               65.147s ( 48.5%)  ##############################
05b. Training - Model Forward/Backward (cumulative)   56.907s ( 42.3%)  ##########################
05a. Training - Neighbor Sampling (cumulative)    6.180s (  4.6%)  ##
05c. Training - Validation Pass (cumulative)    2.082s (  1.5%)  
03d. Feature Engineering - Account Context Features    1.838s (  1.4%)  
02. Transaction Graph Construction (NetworKit)    0.724s (  0.5%)  
03c. Feature Engineering - Transaction Flow Features    0.669s (  0.5%)  
06. Model Inference                       0.415s (  0.3%)  
01. Data Loading & Normalization          0.183s (  0.1%)  
03b. Feature Engineering - Pair History Features    0.094s (  0.1%)  
03a. Feature Engineering - Temporal Features    0.067s (  0.0%)  
04. Graph Export to PyTorch Geometric     0.053s (  0.0%)  
07. Account Risk Aggregation              0.015s (  0.0%)  
----

,stage,calls,total_seconds,mean_seconds,max_seconds,pct_of_total
0,05. Model Training (total),1,65.146628,65.146628,65.146628,48.5
1,05b. Training - Model Forward/Backward (cumula...,536,56.907438,0.106171,0.429680,42.3
2,05a. Training - Neighbor Sampling (cumulative),662,6.179633,0.009335,0.016733,4.6
3,05c. Training - Validation Pass (cumulative),96,2.082470,0.021692,0.025278,1.5
4,03d. Feature Engineering - Account Context Fea...,2,1.838475,0.919237,1.501451,1.4
5,02. Transaction Graph Construction (NetworKit),2,0.723530,0.361765,0.435111,0.5
6,03c. Feature Engineering - Transaction Flow Fe...,2,0.669380,0.334690,0.523553,0.5
7,06. Model Inference,2,0.414512,0.207256,0.210428,0.3
8,01. Data Loading & Normalization,2,0.183395,0.091698,0.129849,0.1
9,03b. Feature Engineering - Pair History Features,2,0.093930,0.046965,0.073990,0.1


## 12. Dashboard  (PRD Section 16)

This notebook just exported `transaction_view.csv`, `account_view.csv`,
and `timing_log.json`. From a terminal in this same folder:

```bash
streamlit run dashboard.py
```

The dashboard has a **Transaction View** tab, an **Account View** tab,
and a bonus **Pipeline Performance** tab that re-renders the timing chart
above (handy for a quick re-check after you swap in the real IBM-AML CSVs
and re-run at full scale).